# ANAADHI — Kaggle GPU Movie Shot Generator

GitHub-managed, Android-friendly notebook. Do not manually edit code unless asked.

**Current target:** `SC001_SH001` — Scene 001 exterior operation establishing shot.

Use a **Tesla T4** accelerator. P100 is intentionally rejected because the current Kaggle/PyTorch stack can fail on it.


In [ ]:
import sys, subprocess, importlib.util

required = ['diffusers', 'transformers', 'accelerate', 'safetensors', 'peft']
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Required AI packages already available — skipping pip download.')

import torch
from pathlib import Path
from PIL import Image
from diffusers import AutoPipelineForText2Image
from IPython.display import display

assert torch.cuda.is_available(), 'GPU is not enabled. Kaggle Settings → Accelerator → GPU T4.'
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
print('CUDA:', torch.version.cuda)
if 'P100' in GPU_NAME.upper():
    raise RuntimeError('P100 is not supported by this notebook. Switch Kaggle Accelerator to GPU T4 and rerun.')


## Locked SC001_SH001 configuration

Scene 001 requires black rain, an ancient Karnataka Western Ghats forest, a raised wooden cabin under covert encirclement, hidden sensor stakes, black-kite police drones, Paraane medical transports, Kendhalaa Police and Sarjanya perimeter forces.

The prompt is deliberately compact so SDXL keeps the important objects inside its CLIP token limit.


In [ ]:
SHOT_ID = 'SC001_SH001'
MODEL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
PROMPT = 'photoreal cinematic pre-dawn Karnataka Western Ghats, rugged raised timber forest cabin on stilts centered and dominant, dense ancient trees, giant wet roots, black monsoon rain, laterite mud, mist, Kendhalaa police perimeter around cabin, Sarjanya rear silhouettes, camouflaged Paraane medical transports under areca leaves, black kite-shaped drones overhead, covert operation, anamorphic wide shot'
NEGATIVE_PROMPT = 'empty forest, empty field, resort, lodge, thatched villa, open veranda, sunny daylight, bright lawn, tiny cabin, hidden cabin, cyberpunk city, American police cars, firefight, explosion, fantasy, anime, illustration, text, watermark, subtitles, black bars, blurry'
SEEDS = [1101, 1102, 1103, 1104]
STEPS = 32
GUIDANCE = 7.0
GEN_WIDTH = 1344
GEN_HEIGHT = 640
MASTER_WIDTH = 3840
MASTER_HEIGHT = 1608
OUTPUT_DIR = Path('/kaggle/working/anaadhi_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Shot:', SHOT_ID)
print('Candidate seeds:', SEEDS)


## Load SDXL

The first model load is the slow part. Keep this Kaggle session open; candidate renders after loading are much faster.


In [ ]:
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
print('Model ready:', MODEL_ID)


## Generate four SC001_SH001 candidates

The notebook now renders four alternatives in one run so you do not have to keep editing or re-importing for every seed. Each output is centre-cropped to the same practical 2.39:1 ratio as the 3840×1608 movie master.


In [ ]:
target_ratio = MASTER_WIDTH / MASTER_HEIGHT
saved = []
for seed in SEEDS:
    print(f'Generating candidate seed {seed}...')
    generator = torch.Generator(device='cpu').manual_seed(seed)
    image = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        width=GEN_WIDTH,
        height=GEN_HEIGHT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        generator=generator,
    ).images[0]

    new_h = round(image.width / target_ratio)
    if new_h <= image.height:
        y0 = (image.height - new_h) // 2
        scoped = image.crop((0, y0, image.width, y0 + new_h))
    else:
        new_w = round(image.height * target_ratio)
        x0 = (image.width - new_w) // 2
        scoped = image.crop((x0, 0, x0 + new_w, image.height))

    out_path = OUTPUT_DIR / f'{SHOT_ID}_seed{seed}.png'
    scoped.save(out_path)
    saved.append(out_path)
    print('Saved:', out_path, 'size:', scoped.size, 'ratio:', round(scoped.width / scoped.height, 4))
    display(scoped)

print('Finished', len(saved), 'candidates.')


## Next

Send the four candidate images/screenshots to ChatGPT. Do not edit Python on the phone. ChatGPT will approve one or revise the GitHub notebook for the next pass.
